# 13 Threshold Profiles

**Purpose.** Threshold-profile generation and evaluation.

**Thesis reference.** §5 Representation Comparison

**Inputs / outputs.** See the data-layout table in the repository README. Paths near the top of the notebook (e.g. `PROFILE_DIR`, `IMAGE_ROOT`, `LABELS`) are relative to the repository root and assume the folder layout described there.

---

In [1]:
import argparse
import json
import time
from pathlib import Path

import numpy as np
import pandas as pd
from scipy.stats import mannwhitneyu, norm
from sklearn.base import BaseEstimator
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score
from sklearn.model_selection import RepeatedStratifiedKFold
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

SEED = 42
MZ_MIN, MZ_MAX = 50.0, 1650.0
RT_MIN, RT_MAX = 0.0, 37.0
MZ_RES = 0.02
RT_BINS, COARSE_MZ = 64, 512
THRESHOLDS = [0, 2, 5, 10]          # 0 = current pipeline, no filtering

# STAGE 1 — generate profiles at every threshold in a single pass
def generate(args):
    root = Path(args.data_root)
    out = Path(args.out_dir)
    out.mkdir(parents=True, exist_ok=True)

    wanted = None
    if args.labels:
        with open(args.labels) as fh:
            wanted = set(json.load(fh).get("labels"))

    files = sorted(root.glob("*.tsv"))
    pairs = []
    for f in files:
        pid = f.stem.replace(args.file_prefix, "")
        if wanted is None or pid in wanted:
            pairs.append((pid, f))
    if args.limit:
        pairs = pairs[:args.limit]
    print(f"Files to process: {len(pairs)}")
    print(f"Thresholds: {THRESHOLDS}  (0 = no filtering, current pipeline)\n")

    n_mz = int(round((MZ_MAX - MZ_MIN) / MZ_RES))
    prof = {t: np.zeros((len(pairs), n_mz), np.float32) for t in THRESHOLDS}
    coarse = {t: np.zeros((len(pairs), RT_BINS, COARSE_MZ), np.float32)
              for t in THRESHOLDS}
    kept = {t: [] for t in THRESHOLDS}
    pids = []

    t0 = time.time()
    for i, (pid, path) in enumerate(pairs):
        n_rows = 0
        acc_p = {t: np.zeros(n_mz, np.float64) for t in THRESHOLDS}
        acc_c = {t: np.zeros((RT_BINS, COARSE_MZ), np.float64)
                 for t in THRESHOLDS}
        n_keep = {t: 0 for t in THRESHOLDS}

        reader = pd.read_csv(path, sep="\t", skiprows=1,
                             names=["time", "mz", "intensity"],
                             dtype=np.float32, chunksize=args.chunk_rows)
        for ch in reader:
            mz = ch["mz"].values
            rt = ch["time"].values
            it = ch["intensity"].values
            n_rows += len(it)

            ok = (mz >= MZ_MIN) & (mz < MZ_MAX) & (rt >= RT_MIN) & (rt < RT_MAX)
            mz, rt, it = mz[ok], rt[ok], it[ok]

            i_fine = ((mz - MZ_MIN) / MZ_RES).astype(np.int64)
            i_cm = ((mz - MZ_MIN) / (MZ_MAX - MZ_MIN) * COARSE_MZ).astype(np.int64)
            i_rt = ((rt - RT_MIN) / (RT_MAX - RT_MIN) * RT_BINS).astype(np.int64)
            np.clip(i_fine, 0, n_mz - 1, out=i_fine)
            np.clip(i_cm, 0, COARSE_MZ - 1, out=i_cm)
            np.clip(i_rt, 0, RT_BINS - 1, out=i_rt)
            flat = i_rt * COARSE_MZ + i_cm

            for t in THRESHOLDS:
                m = it > t if t > 0 else slice(None)
                itm = it[m] if t > 0 else it
                if len(itm) == 0:
                    continue
                n_keep[t] += len(itm)
                acc_p[t] += np.bincount(i_fine[m] if t > 0 else i_fine,
                                        weights=itm, minlength=n_mz)
                acc_c[t] += np.bincount(flat[m] if t > 0 else flat,
                                        weights=itm,
                                        minlength=RT_BINS * COARSE_MZ
                                        ).reshape(RT_BINS, COARSE_MZ)

        for t in THRESHOLDS:
            prof[t][i] = acc_p[t]
            coarse[t][i] = acc_c[t]
            kept[t].append(n_keep[t] / max(n_rows, 1))
        pids.append(pid)

        if (i + 1) % 10 == 0 or i == len(pairs) - 1:
            el = time.time() - t0
            eta = el / (i + 1) * (len(pairs) - i - 1)
            print(f"  [{i+1:4d}/{len(pairs)}] {pid}  "
                  f"{el/60:5.1f} min elapsed, {eta/60:5.1f} min left")

    for t in THRESHOLDS:
        d = out / f"t{t}"
        d.mkdir(exist_ok=True)
        np.save(d / "mz_profiles.npy", prof[t])
        np.save(d / "coarse_maps.npy", coarse[t])
        np.save(d / "mz_centres.npy",
                MZ_MIN + (np.arange(n_mz) + 0.5) * MZ_RES)
        with open(d / "profile_pids.json", "w") as fh:
            json.dump(pids, fh)
        print(f"t={t:2d}: rows retained {np.mean(kept[t]):.1%}  -> {d}")

    with open(out / "sweep_meta.json", "w") as fh:
        json.dump({"thresholds": THRESHOLDS, "n_patients": len(pids),
                   "rows_retained": {str(t): float(np.mean(kept[t]))
                                     for t in THRESHOLDS}}, fh, indent=2)
    print(f"\nDone in {(time.time()-t0)/60:.1f} min")

# STAGE 2 — evaluate each threshold against permutation AND patient ID
class VarianceCap(BaseEstimator):
    def __init__(self, k=3000):
        self.k = k

    def fit(self, X, y=None):
        self.keep_ = (np.arange(X.shape[1]) if X.shape[1] <= self.k
                      else np.sort(np.argsort(X.var(axis=0))[::-1][:self.k]))
        return self

    def transform(self, X):
        return X[:, self.keep_]

def model(k=3000):
    return Pipeline([("cap", VarianceCap(k)), ("sc", StandardScaler()),
                     ("clf", LogisticRegression(C=0.1, max_iter=3000,
                                                class_weight="balanced"))])

def cv_auc(X, y, n_repeats=5, seed=SEED, k=3000):
    rskf = RepeatedStratifiedKFold(n_splits=5, n_repeats=n_repeats,
                                   random_state=seed)
    a = []
    for tr, te in rskf.split(X, y):
        if len(np.unique(y[te])) < 2:
            continue
        try:
            p = model(k).fit(X[tr], y[tr]).predict_proba(X[te])[:, 1]
            a.append(roc_auc_score(y[te], p))
        except Exception:
            continue
    return float(np.mean(a)) if a else 0.5

def tic_log(X):
    t = X.sum(axis=1, keepdims=True)
    t[t == 0] = 1
    return np.log1p(X / t * 1e6)

def evaluate(args):
    base = Path(args.profiles_dir)
    label_files = sorted(Path(args.labels_dir).glob("labels_*.json"))
    label_files += [Path(f) for f in args.extra_labels if Path(f).exists()]

    print(f"Thresholds: {THRESHOLDS}\nLabel files: {len(label_files)}")
    print(f"Permutations: {args.permutations}\n")

    rows = []
    for lf in label_files:
        with open(lf) as fh:
            labels = json.load(fh).get("labels")
        task = lf.stem.replace("labels_", "")
        print(f"\n=== {task} ===")
        print(f"{'thr':>4} {'rows':>7} {'features':12s} {'model':>7} "
              f"{'perm':>7} {'ID':>7} {'margin':>8} {'p(z)':>8}")
        print("-" * 70)

        for t in THRESHOLDS:
            d = base / f"t{t}"
            if not (d / "mz_profiles.npy").exists():
                continue
            with open(d / "profile_pids.json") as fh:
                allp = json.load(fh)
            pids = [p for p in allp if p in labels]
            if len(pids) < 40:
                continue
            idx = [allp.index(p) for p in pids]
            y = np.array([labels[p] for p in pids])
            if min(np.bincount(y)) < 15:
                continue
            ids = np.array([int("".join(c for c in p if c.isdigit()))
                            for p in pids])

            # patient-ID baseline on exactly this cohort
            u, _ = mannwhitneyu(ids[y == 1], ids[y == 0])
            id_auc = u / ((y == 1).sum() * (y == 0).sum())
            id_auc = max(id_auc, 1 - id_auc)

            F = np.load(d / "mz_profiles.npy")[idx].astype(np.float64)
            C = np.load(d / "coarse_maps.npy")[idx].astype(np.float64)
            sets = {"profile_1d": tic_log(F),
                    "map_2d": tic_log(C.reshape(len(C), -1))}

            for fname, X in sets.items():
                obs = cv_auc(X, y, n_repeats=args.cv_repeats,
                             k=args.max_features)
                rng = np.random.default_rng(SEED)
                perms = [cv_auc(X, rng.permutation(y), n_repeats=1,
                                seed=1000 + i, k=args.max_features)
                         for i in range(args.permutations)]
                pm, ps = float(np.mean(perms)), float(np.std(perms, ddof=1))
                z = (obs - pm) / ps if ps > 1e-9 else 0.0
                zp = float(norm.sf(z)) if z > 0 else 1.0
                margin = obs - id_auc
                print(f"{t:4d} {'':7s} {fname:12s} {obs:7.4f} {pm:7.4f} "
                      f"{id_auc:7.4f} {margin:+8.4f} {zp:8.4f}")
                rows.append(dict(task=task, threshold=t, features=fname,
                                 observed=obs, permuted=pm, id_auc=id_auc,
                                 margin=margin, z_p=zp))

    print("\n" + "=" * 74)
    print("Does thresholding improve the margin over patient ID?\n")
    tasks = sorted({r["task"] for r in rows})
    for task in tasks:
        sub = [r for r in rows if r["task"] == task]
        base_m = {r["features"]: r["margin"] for r in sub if r["threshold"] == 0}
        print(f"{task}")
        for f in sorted({r['features'] for r in sub}):
            line = f"  {f:12s}"
            for t in THRESHOLDS:
                m = [r["margin"] for r in sub
                     if r["threshold"] == t and r["features"] == f]
                if m:
                    d = m[0] - base_m.get(f, m[0])
                    line += f"  t{t}: {m[0]:+.3f} ({d:+.3f})"
            print(line)
        print()

    improved = [r for r in rows if r["threshold"] > 0]
    if improved:
        best = max(improved, key=lambda r: r["margin"])
        print(f"Largest margin at any threshold: {best['margin']:+.4f} "
              f"({best['task']}, {best['features']}, t={best['threshold']})")
        base_same = [r for r in rows if r["threshold"] == 0
                     and r["task"] == best["task"]
                     and r["features"] == best["features"]]
        if base_same:
            delta = best["margin"] - base_same[0]["margin"]
            print(f"  versus unfiltered: {delta:+.4f}")
            if delta > 0.05:
                print("  -> Thresholding materially improves the margin. "
                      "Regenerate the full dataset with this filter.")
            else:
                print("  -> Thresholding does not materially change the margin. "
                      "Background was not the limiting factor; report this.")

    with open(args.out, "w") as fh:
        json.dump(rows, fh, indent=2)
    print(f"\nWritten to {args.out}")

def main():
    ap = argparse.ArgumentParser()
    sub = ap.add_subparsers(dest="cmd", required=True)

    g = sub.add_parser("generate")
    g.add_argument("--data-root", required=True)
    g.add_argument("--out-dir", default="threshold_profiles")
    g.add_argument("--labels", default=None,
                   help="restrict to patients in this label file (much faster)")
    g.add_argument("--file-prefix", default="WCmsdata_s")
    g.add_argument("--chunk-rows", type=int, default=4_000_000)
    g.add_argument("--limit", type=int, default=None)
    g.set_defaults(func=generate)

    e = sub.add_parser("evaluate")
    e.add_argument("--profiles-dir", default="threshold_profiles")
    e.add_argument("--labels-dir", default="labels")
    e.add_argument("--extra-labels", nargs="*", default=["labels_id_matched.json"])
    e.add_argument("--out", default="threshold_sweep.json")
    e.add_argument("--permutations", type=int, default=20)
    e.add_argument("--cv-repeats", type=int, default=5)
    e.add_argument("--max-features", type=int, default=3000)
    e.set_defaults(func=evaluate)

    args = ap.parse_args()
    args.func(args)

In [2]:
DATA_ROOT   = "../ahen_data/filtered_tsv_2"
OUT_DIR     = "threshold_profiles"
LABELS      = "labels/labels_main_any_ckd.json"  # restricts to 303 patients
FILE_PREFIX = "WCmsdata_s"
LIMIT       = None      # set to e.g. 40 for a quick timing check

GEN_ARGS = ["generate", "--data-root", DATA_ROOT, "--out-dir", OUT_DIR,
            "--labels", LABELS, "--file-prefix", FILE_PREFIX]
if LIMIT: GEN_ARGS += ["--limit", str(LIMIT)]

In [ ]:
import sys
sys.argv = ["11_threshold_sweep.py"] + GEN_ARGS
main()

Files to process: 303
Thresholds: [0, 2, 5, 10]  (0 = no filtering, current pipeline)

  [  10/303] P0045    3.0 min elapsed,  86.9 min left
  [  20/303] P0079    6.4 min elapsed,  90.9 min left
  [  30/303] P0115    9.8 min elapsed,  89.0 min left
  [  40/303] P0168   13.1 min elapsed,  85.9 min left
  [  50/303] P0236   16.6 min elapsed,  84.0 min left
  [  60/303] P0506   20.5 min elapsed,  83.2 min left
  [  70/303] P0518   24.4 min elapsed,  81.3 min left
  [  80/303] P0545   28.1 min elapsed,  78.4 min left
  [  90/303] P0571   31.9 min elapsed,  75.5 min left
  [ 100/303] P0600   35.7 min elapsed,  72.6 min left
  [ 110/303] P0620   39.5 min elapsed,  69.2 min left
  [ 120/303] P0642   43.0 min elapsed,  65.5 min left
  [ 130/303] P0660   46.8 min elapsed,  62.2 min left
  [ 140/303] P0675   50.6 min elapsed,  58.9 min left
  [ 150/303] P0699   54.3 min elapsed,  55.4 min left
  [ 160/303] P0735   58.1 min elapsed,  51.9 min left
  [ 170/303] P0760   62.1 min elapsed,  48.6 min 